In [ ]:
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report

from claimlens.data import load_data


In [41]:
df_full = pd.read_parquet('../data/processed/abcd_first_msgs.parquet')
# df_sample = pd.read_json('../raw_data/abcd_sample.json')


In [42]:
df = df_full.rename(columns={
    "id": "claim_id",
    "first_msgs": "cleaned_text",
    "refund": "refund_request",
})
df.head()

,split,claim_id,flow,subflow,cleaned_text,refund_request,n_words
0,train,3592,product_defect,return_size,"Hi! I need to return an item, can you help me ...",0,13
1,train,9489,product_defect,refund_status,just wanted to check on the status of a refund,0,10
2,train,3695,storewide_query,timing_4,HEY HO! I've got a promo code and I want to kn...,0,27
3,train,5798,shipping_issue,manage,"Hello, I would like to change my shipping deat...",0,18
4,train,3647,storewide_query,pricing_3,Hello. I have a really cool party coming up. A...,0,34


In [52]:
pd.set_option("display.max_colwidth", None)   # show full text in tables (run once)

def show_claims(subflow, n=20):
    """Show up to n claims for one subflow, with their refund label."""
    rows = df_full[df_full["subflow"] == subflow]       # all chats with this subflow

    refund_rate = rows["refund"].mean()                 # share of chats that were refunded
    print(f"{subflow}: {len(rows)} chats, {refund_rate:.0%} refunded")

    return rows[["first_msgs", "refund"]].head(n)       # table with just the text and the label

show_claims("return_size")

return_size: 191 chats, 2% refunded


,first_msgs,refund
0,"Hi! I need to return an item, can you help me with that?",0
19,hi i would like to return an item,0
84,I ordered a pair of jeans in the wrong size and would like to return them,0
98,I would like to return an item because it is the wrong size,0
120,hi I need to order a pair of boots,0
125,Hi there I need to make a return i got a size to big,0
153,"Hello, I want to return an item. It's too big for me.",0
208,I need to return an item that is the wrong size,0
269,Hi i want to return my order i ordered the wrong size jacket,0
343,"Hi, I ordered these jeans for my daughter, and they're too small.",0


In [43]:
REFUND_SUBFLOWS = {
    "refund_initiate",
    "return_size",
    "return_color",
    "return_stain",
    "manage_dispute_bill",
    "bad_price_competitor",
    "status_mystery_fee",
    "refund_update",
    "manage_cancel",
}

df["refund_request"] = df["subflow"].isin(REFUND_SUBFLOWS).astype(int)
print(df["refund_request"].value_counts())

refund_request
0    8557
1    1485
Name: count, dtype: int64


In [44]:
train = df[df["split"] == 'train']
val = df[df["split"] == 'dev']
print(train)
print(val)


      split  claim_id               flow                subflow  \
0     train      3592     product_defect            return_size   
1     train      9489     product_defect          refund_status   
2     train      3695    storewide_query               timing_4   
3     train      5798     shipping_issue                 manage   
4     train      3647    storewide_query              pricing_3   
...     ...       ...                ...                    ...   
8029  train      2229  troubleshoot_site         search_results   
8030  train      6109     manage_account  status_credit_missing   
8031  train      9892  single_item_query            jeans_how_2   
8032  train      4036  single_item_query          jeans_other_2   
8033  train      5882  troubleshoot_site         search_results   

                                           cleaned_text  refund_request  \
0     Hi! I need to return an item, can you help me ...               1   
1        just wanted to check on the status o

In [45]:
X_train = train['cleaned_text']
y_train = train['refund_request']

X_val = val['cleaned_text']
y_val = val['refund_request']

In [ ]:
models = {
    "count + none": make_pipeline(
        CountVectorizer(),
        LogisticRegression(max_iter=1000)
    ),
    "count + balanced": make_pipeline(
        CountVectorizer(),
        LogisticRegression(max_iter=1000, class_weight="balanced")
    ),
    "tfidf + none": make_pipeline(
        TfidfVectorizer(),
        LogisticRegression(max_iter=1000)
    ),
    "tfidf + balanced": make_pipeline(
        TfidfVectorizer(),
        LogisticRegression(max_iter=1000, class_weight="balanced")
    )
}

In [ ]:
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    print(f"\n=== {name} ===")
    print(classification_report(y_val, pipe.predict(X_val)))

In [46]:
# df_full['flow'] = df_full['scenario'].str['flow']
# df_full['subflow'] = df_full['scenario'].str['subflow']

In [47]:
# def quick_claim_text(turns, n=3):
#     """Rough version: first n customer messages joined. Temporary until the group's cleaned column is ready."""
#     msgs = [t["text"] for t in turns if t["speaker"] == "customer"]   # customer turns only
#     return " ".join(msgs[:n])                                         # first n, joined with spaces

# df_full["claim_text"] = df_full["delexed"].apply(quick_claim_text)
# df_full["subflow"] = df_full["scenario"].str["subflow"]

In [48]:
# '''Checking if Dataframe is received correctly'''
# REQUIRED = ["claim_id", "claim_text", "subflow", "wants_refund", "split"]
# missing = set(REQUIRED) - set(df.columns)
# assert not missing, f"Missing columns: {missing}"

# assert df["claim_id"].is_unique, "Duplicate claim IDs"

# df["wants_refund"] = Rdf["wants_refund"].astype(int)
# print("empty texts:", (df["claim_text"].fillna("").str.strip() == "").sum())
# print("missing subflows:", df["subflow"].isna().sum())
# print(df["split"].value_counts())

In [49]:
# REFUND_SUBFLOWS = set(df.loc[df["wants_refund"] == 1, "subflow"])
# sorted(REFUND_SUBFLOWS)

In [50]:
# df_full["subflow"].value_counts()